# Chapter 3: Forward Propagation — Intro to Neural Prediction
### Grokking Deep Learning (Andrew W. Trask)
**Student Name**: Muhammad Aidil Fadhli Awalludin  
**NIM**: 101032300087  
**Class**: TK4701  

---

> *"Prediction is very difficult, especially if it's about the future."*  
> — Niels Bohr

---

## 1. Executive Summary & Objective

In this chapter, we unpack the first pillar of the deep learning paradigm: **Predict**. Forward propagation is the process through which a neural network accepts sensory inputs, transforms them through a network of weighted connections, and outputs a prediction.

### Core Objectives:
1. **Conceptualize Neural Weights**: Understand weights as directional "volume knobs" that scale, invert, or mute information.
2. **Progressive Architecture Scaling**: Build prediction networks across five distinct topologies:
   - Single Input $\to$ Single Output
   - Multiple Inputs $\to$ Single Output (Weighted Sum)
   - Single Input $\to$ Multiple Outputs (Signal Broadcasting)
   - Multiple Inputs $\to$ Multiple Outputs (Full Matrix Transformation)
   - Deep Networks (Input $\to$ Hidden Layer $\to$ Output Layer)
3. **Dual Implementation**: Code each network using raw Python primitives (`lists`, `loops`) to demystify data flow, alongside modern vectorized `NumPy` implementations.

---
## 2. Theoretical Deep-Dive

### 2.1 The Anatomy of Neural Prediction

A neural network prediction consists of three fundamental entities:
1. **Input Data ($x$)**: Recorded measurements or observations from the real world (e.g., player stats, sensor readings, pixels).
2. **Weights ($w$)**: The internal parameters of the network. A weight measures the *sensitivity* or *correlation* between an input signal and the output node.
   - $w > 0$: Positive correlation (an increase in input increases the prediction).
   - $w < 0$: Negative correlation (an increase in input decreases the prediction).
   - $w = 0$: Complete irrelevance (the input is ignored).
3. **Prediction ($\hat{y}$)**: The generated hypothesis computed by the network.

```
      Input Node (x) ------------[ Weight w ]------------> Output Node (y_pred)
                                y_pred = x * w
```

### 2.2 Mathematical Formulations of Prediction Topologies

#### 1. Multiple Inputs $\to$ Single Output
When a single output neuron receives multiple signals $\mathbf{x} = [x_1, x_2, \dots, x_m]^T$ through weights $\mathbf{w} = [w_1, w_2, \dots, w_m]^T$:

$$\hat{y} = \sum_{i=1}^m x_i w_i = \mathbf{x}^T \mathbf{w}$$

This is mathematically equivalent to the **dot product** of the input vector and the weight vector.

#### 2. Single Input $\to$ Multiple Outputs
When a single observation $x$ informs multiple independent predictions $\hat{\mathbf{y}} = [\hat{y}_1, \dots, \hat{y}_k]^T$ via weights $\mathbf{w} = [w_1, \dots, w_k]^T$:

$$\hat{y}_j = x \cdot w_j \quad \implies \quad \hat{\mathbf{y}} = x \mathbf{w}$$

#### 3. Multiple Inputs $\to$ Multiple Outputs
When $m$ inputs map to $k$ outputs, the weights form a 2D matrix $\mathbf{W} \in \mathbb{R}^{k \times m}$:

$$\begin{bmatrix} \hat{y}_1 \\ \hat{y}_2 \\ \vdots \\ \hat{y}_k \end{bmatrix} = 
\begin{bmatrix} 
w_{11} & w_{12} & \dots & w_{1m} \\ 
w_{21} & w_{22} & \dots & w_{2m} \\ 
\vdots & \vdots & \ddots & \vdots \\ 
w_{k1} & w_{k2} & \dots & w_{km} 
\end{bmatrix} 
\begin{bmatrix} x_1 \\ x_2 \\ \vdots \\ x_m \end{bmatrix} 
\quad \implies \quad \hat{\mathbf{y}} = \mathbf{W} \mathbf{x}$$

#### 4. Stacking Layers: Deep Networks (Predicting on Predictions)
In deep networks, intermediate hidden layers $\mathbf{h}$ act as learned representations:

$$\mathbf{h} = \mathbf{W}_{ih} \mathbf{x}, \quad \hat{\mathbf{y}} = \mathbf{W}_{hp} \mathbf{h} = \mathbf{W}_{hp} (\mathbf{W}_{ih} \mathbf{x})$$

Here, $\mathbf{W}_{ih}$ represents input-to-hidden weights, and $\mathbf{W}_{hp}$ represents hidden-to-prediction weights.

---
## 3. Code Reproduction

We now reproduce each architecture using Andrew Trask's canonical baseball dataset:

In [ ]:
# 3.1 Architecture 1: Simplest Neural Network (Single Input, Single Output)

weight = 0.1

def neural_network_single(number_of_toes, weight):
    prediction = number_of_toes * weight
    return prediction

number_of_toes = [8.5, 9.5, 10.0, 9.0]
input_data = number_of_toes[0]

pred = neural_network_single(input_data, weight)
print(f"Single Input ({input_data}) * Weight ({weight}) = Prediction: {pred:.4f}")

In [ ]:
# 3.2 Architecture 2: Multiple Inputs, Single Output (Pure Python w_sum)

def w_sum(a, b):
    assert len(a) == len(b), "Input and weights must have identical length!"
    output = 0.0
    for i in range(len(a)):
        output += (a[i] * b[i])
    return output

weights = [0.1, 0.2, 0.0]  # [weight_toes, weight_win_loss, weight_fans]

def neural_network_multi_input(inputs, weights):
    prediction = w_sum(inputs, weights)
    return prediction

# Baseball dataset: game 1
toes =  [8.5, 9.5, 9.9, 9.0]
wlrec = [0.65, 0.8, 0.8, 0.9]
nfans = [1.2, 1.3, 0.5, 1.0]

input_game_1 = [toes[0], wlrec[0], nfans[0]]
pred = neural_network_multi_input(input_game_1, weights)

print(f"Input Features: {input_game_1}")
print(f"Weights:        {weights}")
print(f"Win Probability Prediction (Scratch): {pred:.4f}")

In [ ]:
# 3.3 Architecture 2 with NumPy Vectorization
import numpy as np

weights_np = np.array([0.1, 0.2, 0.0])
inputs_np = np.array([toes[0], wlrec[0], nfans[0]])

def neural_network_multi_input_np(inputs, weights):
    return inputs.dot(weights)

pred_np = neural_network_multi_input_np(inputs_np, weights_np)
print(f"Win Probability Prediction (NumPy):   {pred_np:.4f}")
assert np.isclose(pred, pred_np)

In [ ]:
# 3.4 Architecture 3: Single Input, Multiple Outputs
# Predicting [hurt?, win?, sad?] from win/loss record alone

def ele_mul(scalar, vector):
    output = [0.0] * len(vector)
    for i in range(len(vector)):
        output[i] = scalar * vector[i]
    return output

weights_multi_out = [0.3, 0.2, 0.9]  # [w_hurt, w_win, w_sad]
input_record = wlrec[0]             # 0.65

def neural_network_single_to_multi(input_val, weights):
    return ele_mul(input_val, weights)

pred_multi = neural_network_single_to_multi(input_record, weights_multi_out)
labels = ["Hurt Prob", "Win Prob", "Sad Prob"]
for label, val in zip(labels, pred_multi):
    print(f"{label:10s}: {val:.4f}")

In [ ]:
# 3.5 Architecture 4: Multiple Inputs, Multiple Outputs (Full Matrix)

# Weights mapping (toes, wlrec, nfans) -> (hurt, win, sad)
weights_matrix = [
    [0.1, 0.1, -0.3],  # hurt?
    [0.1, 0.2, 0.0],   # win?
    [0.0, 1.3, 0.1]    # sad?
]

def vect_mat_mul(vector, matrix):
    assert len(vector) == len(matrix[0])
    output = [0.0] * len(matrix)
    for i in range(len(matrix)):
        output[i] = w_sum(vector, matrix[i])
    return output

def neural_network_full(inputs, weights):
    return vect_mat_mul(inputs, weights)

preds_full_scratch = neural_network_full(input_game_1, weights_matrix)
preds_full_np = np.array(weights_matrix).dot(np.array(input_game_1))

print("Multiple Input -> Multiple Output Predictions:")
for label, s_val, n_val in zip(labels, preds_full_scratch, preds_full_np):
    print(f"{label:10s} | Scratch: {s_val:8.4f} | NumPy: {n_val:8.4f}")
assert np.allclose(preds_full_scratch, preds_full_np)

In [ ]:
# 3.6 Architecture 5: Stacking Layers (Deep Neural Network: Hidden Layer)

# Input-to-hidden weights (3 inputs -> 3 hidden units)
ih_wgt = [
    [0.1, 0.2, -0.1],  # hid[0]
    [-0.1, 0.1, 0.9],  # hid[1]
    [0.1, 0.4, 0.1]    # hid[2]
]

# Hidden-to-prediction weights (3 hidden units -> 3 outputs)
hp_wgt = [
    [0.3, 1.1, -0.3],  # hurt?
    [0.1, 0.2, 0.0],   # win?
    [0.0, 1.3, 0.1]    # sad?
]

def neural_network_deep(inputs, ih_wgt, hp_wgt):
    # Forward pass layer 1 (input -> hidden)
    hid = vect_mat_mul(inputs, ih_wgt)
    # Forward pass layer 2 (hidden -> output)
    pred = vect_mat_mul(hid, hp_wgt)
    return hid, pred

hidden_acts, deep_preds = neural_network_deep(input_game_1, ih_wgt, hp_wgt)

# NumPy equivalent
hid_np = np.array(ih_wgt).dot(np.array(input_game_1))
pred_deep_np = np.array(hp_wgt).dot(hid_np)

print("Deep Network Forward Pass Output:")
print("Hidden Activations: ", np.round(hid_np, 4))
print("Output Predictions: ", np.round(pred_deep_np, 4))
assert np.allclose(deep_preds, pred_deep_np)

---
## 4. Chapter 3 Summary & Key Takeaways

| Concept | Mathematical Form | Theoretical Role |
|:---|:---:|:---|
| **Weights as Knobs** | $\hat{y} = w \cdot x$ | Weights calibrate how strongly an input influences the prediction and in which direction. |
| **Multiple Inputs** | $\hat{y} = \mathbf{w}^T \mathbf{x}$ | Combines evidence from multiple features using the dot product (weighted sum). |
| **Multiple Outputs** | $\hat{\mathbf{y}} = \mathbf{W}\mathbf{x}$ | Transforms input features simultaneously across multiple prediction tasks using matrix multiplication. |
| **Hidden Layers** | $\hat{\mathbf{y}} = \mathbf{W}_{hp}(\mathbf{W}_{ih}\mathbf{x})$ | Creates intermediate latent abstractions ("predicting on predictions"). |
| **Next Step** | In **Chapter 4**, we address the crucial question: *How do we determine the correct values of these weights so the network actually makes accurate predictions?* We dive into **Compare** (loss measurement) and **Learn** (Gradient Descent). |